<a href="https://colab.research.google.com/github/Anshul-ARK/nlp_lab_assignments/blob/main/unigram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from datasets import load_dataset
dt = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", split ="tel_Telu",streaming=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

In [25]:
#first we get the counts of the unigrams
from collections import defaultdict
import re
d = defaultdict(int)
pattern_sentence_tokenization =  r'(?<=[.।!])\s+'
pattern_word_tokenization = r'[\u0C00-\u0C7F]+'
for i in dt:
    lis = re.split(pattern_sentence_tokenization,i['text'])
    # print(i['text'])
    # print(lis)
    # break
    #for unigram no need to add the sentence starting tokens
    for j in lis:
      lis2 = re.findall(pattern_word_tokenization,j)
      for j in lis2:
        d[j] += 1
#here i have to store the default dict values into the csv file


In [26]:
import csv
import os

# Define the path to your file in Google Drive
file_path = '/content/drive/MyDrive/nlp_assignments/unigram_model/unigram_frequencys.csv'

# Ensure the directory exists
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Write to the CSV file
with open(file_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    # Write header row
    writer.writerow(d.keys())
    # Write data row
    writer.writerow(d.values())

print(f"CSV file created at: {file_path}")

CSV file created at: /content/drive/MyDrive/nlp_assignments/unigram_model/unigram_frequencys.csv


In [27]:
import math
def add_one_smoothing(x : str):
  #first we tokenize the sentence
  lis2 = re.findall(r'[\u0C00-\u0C7F]+',x)
  #we calculate the log score instead of the normal product score
  #N = TOTAL NUMBER OF WORDS
  ans = 0
  v = len(d)
  N = 0
  for i in d.values():
    N += i
  for i in lis2:
    val = (d[i] + 1) / (N + v)
    ans += math.log(val)
  return ans

In [28]:
def add_k_smoothing(x : str):
  #the formula was to
  lis2 = re.findall(r'[\u0C00-\u0C7F]+',x)
  #we calculate the log score instead of the normal product score
  #N = TOTAL NUMBER OF WORDS
  ans = 0
  v = len(d)
  N = 0
  k = 0.1
  for i in d.values():
    N += i
  for i in lis2:
    val = (d[i] + k) / (N + k*v)
    ans += math.log(val)
  return ans

In [29]:
def add_token_type_smoothing(x : str):
  lis2 = re.findall(r'[\u0C00-\u0C7F]+',x)
  #we calculate the log score instead of the normal product score
  #N = TOTAL NUMBER OF WORDS
  ans = 0
  v = len(d)
  N = 0
  k = 0.1
  for i in d.values():
    N += i
  #for the unigrams we dont have the type token
  for i in lis2:
    val = (d[i] + k) / (N + k*v)
    ans += math.log(val)
  return ans

First, you'll need to read the sentences from your text file. Make sure to update the `file_path` to the actual location of your file.

In [ ]:
# Define the path to your text file
file_path = '/content/drive/MyDrive/nlp_assignments/training-data-telugu (1).txt' # Update this path
sentences = []
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            # line = line.strip
            sentences.append(line.strip()) # Remove leading/trailing whitespace
            print(line)
            # break
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

print(f"Read {len(sentences)} sentences from the file.")

Now you can iterate through the sentences and apply your smoothing functions.

In [36]:
# Example of applying the smoothing functions to each sentence
for i, sentence in enumerate(sentences):
    print(f"Processing sentence {i+1}: {sentence}")
    # Apply add-one smoothing
    one_smoothing_score = add_one_smoothing(sentence)
    print(f"  Add-one smoothing score: {one_smoothing_score}")

    # Apply add-k smoothing
    k_smoothing_score = add_k_smoothing(sentence)
    print(f"  Add-k smoothing score: {k_smoothing_score}")

    # Apply add-token-type smoothing
    token_type_smoothing_score = add_token_type_smoothing(sentence)
    print(f"  Add-token-type smoothing score: {token_type_smoothing_score}")

    # print("-" * 20) # Separator for clarity

Streaming output truncated to the last 5000 lines.
Processing sentence 552: వివో వి5 ప్లస్ 3055 ఎంఏహెచ్ నాన్-రిమూవేబుల్ బ్యాటరీని కలిగి ఉంది, ఇది ఛార్జ్ చేసిన తర్వాత రోజంతా సులభంగా నడుస్తుంది.
  Add-one smoothing score: -156.85369154109185
  Add-k smoothing score: -157.30901478267748
  Add-token-type smoothing score: -157.30901478267748
Processing sentence 553: మా హెచ్డి వీడియో లూప్ పరీక్షలో మేము ఫోన్ను 10 గంటలు మరియు 53 నిమిషాలు సులభంగా నడపగలిగాము, ఇది పేలవమైన పనితీరు అని చెప్పలేము.
  Add-one smoothing score: -178.62833134855725
  Add-k smoothing score: -180.7861223792006
  Add-token-type smoothing score: -180.7861223792006
Processing sentence 554: వీవో యొక్క'డ్యూయల్-ఛార్జింగ్ ఇంజిన్'టెక్నాలజీతో ఫాస్ట్ ఛార్జింగ్ మద్దతు కూడా ఉంది, అధిక కరెంట్ కోసం రెండు మైక్రోచిప్స్ స్పెసిఫికేషన్లు ఉన్నాయి.
  Add-one smoothing score: -171.2798344566651
  Add-k smoothing score: -171.1436139343278
  Add-token-type smoothing score: -171.1436139343278
Processing sentence 555: దీనితో, మేము పరికరాన్ని అరగంటల